In [ ]:
# bootstrap: Colab clone + local import of `agentcore` (auto-inserted)
import sys, pathlib
if "google.colab" in sys.modules:
    import os, subprocess
    _slug = "aniryou/full-stack-agentic-engineer"
    _repo = pathlib.Path("/content/full-stack-agentic-engineer")
    if not _repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_repo)], check=True)
    os.chdir(_repo / "07-application-agent-framework/agent-fundamentals/agent-core")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
_r = pathlib.Path.cwd().resolve()
while _r != _r.parent and not (_r / "agentcore").exists():
    _r = _r.parent
if str(_r) not in sys.path:
    sys.path.insert(0, str(_r))
del _r

# 03 · State and control

Two things separate a demo loop from something you would run: it remembers the
conversation across turns, and it protects itself — a step budget, and not calling
the same tool over and over. This notebook adds both, and shows a `policy` model that
actually reacts to tool results (so you can drive multi-step behaviour without a rigid
script).

In [ ]:
from agentcore import Agent, FakeLLM, call, calls, text, tool

@tool
def get_balance(account_id: str) -> dict:
    """Return an account balance."""
    return {"account_id": account_id, "balance": 1234.5}

## Multi-turn memory
`Agent.run` returns the full transcript in `result.messages`. Pass it back as
`history` on the next turn and the model sees the earlier exchange.

In [ ]:
agent = Agent(FakeLLM([call("get_balance", account_id="a1"), "It's 1234.5.",
                       "You asked about account a1."]), tools=[get_balance])
first = agent.run("balance for a1?")
second = agent.run("which account did I just ask about?", history=first.messages)
print("second answer:", second.text)
print("history carried", len(second.messages), "messages")

## Exercise 3.1 — a policy that reacts to results

A **policy** decides each turn from the messages: `(messages, tools) -> Response`.
Write `balance_policy` that: if the messages after the last user turn contain a tool
result, answer with `text("Your balance is on file.")`; else if the last user message
mentions `"balance"`, return `call("get_balance", account_id="a1")`; else
`text("How can I help?")`.

In [ ]:
def balance_policy(messages, tools):
    last_user = max((i for i, m in enumerate(messages) if m.get("role") == "user"), default=-1)
    after = messages[last_user + 1:]
    if any(m.get("role") == "tool" for m in after):
        return text("Your balance is on file.")
    if last_user >= 0 and "balance" in str(messages[last_user]["content"]).lower():
        return call("get_balance", account_id="a1")
    return text("How can I help?")

In [ ]:
r = Agent(FakeLLM(policy=balance_policy), tools=[get_balance]).run("what's my balance?")
assert r.done and r.text == "Your balance is on file."
assert [m["role"] for m in r.messages] == ["system", "user", "assistant", "tool", "assistant"]
assert Agent(FakeLLM(policy=balance_policy), tools=[get_balance]).run("hello").text == "How can I help?"
print("✅ policy drives the loop without a fixed script")

## Exercise 3.2 — stop a repeated tool call

A confused model can call the same tool with the same arguments forever, burning
tokens. Write `dedupe(tool_calls, seen)`: `seen` is a set of signatures already run
(use `tc.signature()`). Return only the calls whose signature is **new**, and add
them to `seen`. (In a real loop you would feed the model a "you already have this"
note for the dropped ones; here we just filter.)

In [ ]:
def dedupe(tool_calls, seen: set) -> list:
    fresh = []
    for tc in tool_calls:
        sig = tc.signature()
        if sig not in seen:
            seen.add(sig)
            fresh.append(tc)
    return fresh

In [ ]:
seen = set()
batch = [call("get_balance", account_id="a1"), call("get_balance", account_id="a1"),
         call("get_balance", account_id="a2")]
fresh = dedupe(batch, seen)
assert [tc.args["account_id"] for tc in fresh] == ["a1", "a2"]   # the duplicate a1 is gone
assert dedupe([call("get_balance", account_id="a1")], seen) == []  # already seen
print("✅ dedupe drops repeated calls")

## Exercise 3.3 — reason about the budget

`Agent(max_steps=N)` caps model calls per turn. For a model that **never** answers
(always returns a tool call), how many `assistant` messages appear in the transcript
when `max_steps=4`, and is `result.done` True or False? Set the two variables, then
the check confirms them against a real run.

In [ ]:
expected_assistant_messages = 4
expected_done = False

In [ ]:
runaway = FakeLLM(policy=lambda m, t: calls(call("get_balance", account_id="a1")))
r = Agent(runaway, tools=[get_balance], max_steps=4).run("go")
assert sum(1 for m in r.messages if m["role"] == "assistant") == expected_assistant_messages
assert r.done is expected_done
print(f"✅ {expected_assistant_messages} model calls, done={expected_done} — the budget saved you")

## The one-minute version
"State management" is on the rubric. Name the kinds: the **conversation** (the
transcript), small **working state** (what stage a task is at), and **budgets**
(steps, and in production tokens and time). Say a loop without a step budget is a cost
incident waiting to happen — and that you cap it in code, not in the prompt.